In [1]:
!uv add openai-agents==0.3.3

Resolved 168 packages in 154ms                                       
░░░░░░░░░░░░░░░░░░░░ [0/12] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 12 packages in 75ms                               
 + colorama==0.4.6
 + griffe==1.14.0
 + httpx-sse==0.4.3
 + mcp==1.18.0
 + openai-agents==0.3.3
 + pydantic-settings==2.11.0
 + python-dotenv==1.1.1
 + python-multipart==0.0.20
 + sse-starlette==3.0.2
 + starlette==0.48.0
 + types-requests==2.32.4.20250913
 + uvicorn==0.38.0


In [26]:
from agents import Agent, function_tool, Runner
import requests
import asyncio

In [19]:
import requests
from typing import Optional

def get_page_content(url: str) -> Optional[str]:
    """
    Fetch the content of a given URL through the Jina Reader proxy.

    This function prepends the Jina Reader base URL to the provided URL,
    sends a GET request, and returns the decoded content if successful.

    Args:
        url (str): The target URL to fetch content from.

    Returns:
        Optional[str]: The decoded content of the response if the request is successful;
                       None if an error occurs or the response is not OK.

    Raises:
        ValueError: If the provided URL is empty.
    """
    if not url:
        raise ValueError("The 'url' parameter cannot be empty.")

    jina_base_url = "https://r.jina.ai/"
    jina_reader_url = f"{jina_base_url}{url}"

    try:
        response = requests.get(jina_reader_url, timeout=10)
        response.raise_for_status()  # Raises HTTPError for bad responses (4xx, 5xx)
        return response.content.decode('utf-8')

    except requests.exceptions.RequestException as e:
        # Catch all network-related errors
        print(f"Error fetching URL '{jina_reader_url}': {e}")
        return None


In [ ]:
content = get_page_content("https://datatalks.club")

In [22]:
assistant_instructions = """
You're a helpful assistant that helps answer user questions.
"""

assistant = Agent(
    name='assistant',
    tools=[function_tool(get_page_content)],
    instructions=assistant_instructions,
    model='gpt-4o-mini'
)


In [23]:
runner = Runner()

In [24]:
user_prompt = "Summarize the content of https://openai.github.io/openai-agents-python/"

result = await runner.run(assistant, input=user_prompt)


In [29]:
### NON notebook code below ###
"""
Need this instead of await to execute the async function.


import asyncio
result = asyncio.run(runner.run(assistant, input=user_prompt))



"""

'\nNeed this instead of await to execute the async function.\n\n\nimport asyncio\nresult = asyncio.run(runner.run(assistant, input=user_prompt))\n\n\n\n'

In [28]:
result

RunResult(input='Summarize the content of https://openai.github.io/openai-agents-python/', new_items=[ToolCallItem(agent=Agent(name='assistant', handoff_description=None, tools=[FunctionTool(name='get_page_content', description='Fetch the content of a given URL through the Jina Reader proxy.\n\nThis function prepends the Jina Reader base URL to the provided URL,\nsends a GET request, and returns the decoded content if successful.', params_json_schema={'properties': {'url': {'description': 'The target URL to fetch content from.', 'title': 'Url', 'type': 'string'}}, 'required': ['url'], 'title': 'get_page_content_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x74b706a8fce0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)], mcp_servers=[], mcp_config={}, instructions="\nYou're a helpful assistant that helps answer user questions.\

In [30]:
result.new_items[-1].raw_item.content[0].text


'The **OpenAI Agents SDK** is a framework for building AI applications that utilize agents—essentially LLMs (Large Language Models) equipped with specific instructions and tools. This SDK streamlines the creation of agentic apps, allowing for complex relationships between tools and agents without a steep learning curve.\n\n### Key Features:\n- **Agents**: Core components that perform tasks based on provided instructions.\n- **Handoffs**: Enable agents to delegate specific tasks to other agents.\n- **Guardrails**: Validate inputs and outputs of agents to ensure safe operations.\n- **Sessions**: Automatically maintain conversation history across agent interactions.\n\n### Installation:\nThe SDK can be installed using:\n```bash\npip install openai-agents\n```\n\n### Example Usage:\nA simple example demonstrates creating an agent that can generate a haiku:\n```python\nfrom agents import Agent, Runner\n\nagent = Agent(name="Assistant", instructions="You are a helpful assistant")\nresult = R

In [31]:
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIAgentsSDKRunner

chat_interface = IPythonChatInterface()

runner = OpenAIAgentsSDKRunner(
    chat_interface=chat_interface,
    agent=assistant
)


In [33]:
await runner.run()

Error fetching URL 'https://r.jina.ai/https://www.linkedin.com/in/jpurrutia': HTTPSConnectionPool(host='r.jina.ai', port=443): Read timed out. (read timeout=10)


Error fetching URL 'https://r.jina.ai/https://www.fantasypros.com': HTTPSConnectionPool(host='r.jina.ai', port=443): Read timed out. (read timeout=10)


Chat ended.


In [37]:
from youtube_transcript_api import YouTubeTranscriptApi

def format_timestamp(seconds: float) -> str:
    """Convert seconds to H:MM:SS if > 1 hour, else M:SS"""
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}:{minutes:02}:{secs:02}"
    else:
        return f"{minutes}:{secs:02}"


def make_subtitles(transcript) -> str:
    lines = []

    for entry in transcript:
        ts = format_timestamp(entry.start)
        text = entry.text.replace('\n', ' ')
        lines.append(ts + ' ' + text)

    return '\n'.join(lines)


def fetch_transcript_raw(video_id):
    ytt_api = YouTubeTranscriptApi()
    transcript = ytt_api.fetch(video_id)
    return transcript


def fetch_transcript_text(video_id):
    transcript = fetch_transcript_raw(video_id)
    subtitles = make_subtitles(transcript)
    return subtitles  


In [39]:
from pathlib import Path

def fetch_transcript_cached(video_id):
    cache_dir = Path("../data_cache/youtube_videos")
    cache_file = cache_dir / f"{video_id}.txt"

    if cache_file.exists():
        return cache_file.read_text(encoding="utf-8")

    subtitles = fetch_transcript_text(video_id)
    cache_file.write_text(subtitles, encoding="utf-8")

    return subtitles


In [40]:
subtitles = fetch_transcript_cached('vK_SxyqIfwk')
#print(subtitles[:500])


RequestBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=vK_SxyqIfwk! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have used to aut